## NBA Shot Selection Prediction 

## Business Understanding

Kobe Bryant, one of the greatest basketball players in history, played 20 seasons with the Los Angeles Lakers. Over his career, he took thousands of shots in varying game situations — from buzzer-beaters to playoff clutch moments. Understanding the context and success rate of these shots can offer powerful insights for sports analysts, coaches, and players.

In this project, we analyze the **complete dataset of Kobe Bryant’s shot attempts** to identify patterns that lead to successful or missed shots. The ultimate goal is to **build a machine learning model** that can predict whether a shot is likely to be successful based on game and shot conditions.

##  Dataset Overview

The dataset contains detailed information about **every field goal attempt Kobe Bryant took** during his 20-year NBA career. It includes features like:

- **Game context**: period, time remaining, opponent, playoffs
- **Shot context**: location (x, y, lat, lon), shot distance, shot type, zone
- **Game info**: season, matchup, home/away
- **Target variable**: `shot_made_flag` (0 or 1)

> Total rows: ~30,000 | Features: 25+ | Domain: **Sports Analytics**


## What is `shot_made_flag`?

- **`shot_made_flag` = 1**: The shot was **made** (successful)
- **`shot_made_flag` = 0**: The shot was **missed**
- This is the **target variable** for prediction.

## Problem Type

This is a **Supervised Binary Classification Problem**.

We will use machine learning algorithms to **predict whether a shot will be made (1) or missed (0)** based on the features describing the shot and the game situation.

## The objective is to:
- Analyze Kobe Bryant’s shot data using visual analytics.
- Help stakeholders understand shot selection patterns.
- Predict the probability of a successful shot using machine learning.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = 'notebook' 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import shap

In [ ]:
#loading the dataset
data = pd.read_csv('C:/Users/arbaj/Projects/NBA Shot Predictions/data/data.csv')

In [ ]:
#first 5 rows of the data
data.head()

In [ ]:
#last 5 rows of the data
data.tail()

In [ ]:
#basic information
data.info()

In [ ]:
# Drop rows where target is null
data = data.dropna(subset=['shot_made_flag'])

# Convert shot_made_flag to int for clear coloring
data['shot_made_flag'] = data['shot_made_flag'].astype(int)


In [ ]:
#statistical information
data.describe()

In [ ]:
#checking for null values
data.isnull().sum()

In [ ]:
#insight : we can see that 'shot_made_flag' column which is a target column has the 5000 null values.

## EDA 

## Univariate Analysis

In [ ]:
# ✅ Check class balance
sns.countplot(x='shot_made_flag', data=data)
plt.title('Distribution of Target Variable: shot_made_flag')
plt.show()

✅ What it shows:
This tells us how balanced the target variable is:
1 = Shot made
0 = Shot missed
💡 Insights:
If the classes are imbalanced (say 60% missed, 40% made), certain models might be biased.
This helps decide whether we need to apply balancing techniques like SMOTE, or use class weights in modeling.

In [ ]:
# ✅ Numerical Feature: shot_distance
sns.histplot(data['shot_distance'], bins=30, kde=True)
plt.title("Distribution of Shot Distance")
plt.xlabel("Shot Distance")
plt.ylabel("Count")
plt.show()

✅ What it shows:
This plot shows how far most shots were taken from the basket.
💡 Insights:
You may see a peak near 0–10 feet, showing Kobe often took close-range shots (like layups).
A second cluster may appear at 22-25 feet, which are likely 3-point shots.
Long-distance shots (beyond 30 feet) are rare — possibly buzzer beaters or half-court attempts.

Why it matters:
Distance from the basket strongly impacts scoring probability. This helps in modeling and also in suggesting better shot zones.

In [ ]:
# ✅ Categorical Feature: period
sns.countplot(x='period', data=data)
plt.title("Shot Attempts by Game Period")
plt.xlabel("Period (Quarter)")
plt.ylabel("Number of Shots")
plt.show()

✅ What it shows:
This tells you how many shots were taken in each quarter (period).
💡 Insights:
Most games have 4 periods. If you see activity in period 5 or more, that indicates overtime shots.
You’ll likely see more shots in the 1st and 3rd quarters when starters play longer.

Why it matters:
Some players perform differently in clutch or fatigue situations. Later periods might affect shot accuracy.

In [ ]:
# ✅ Minutes Remaining
sns.histplot(data['minutes_remaining'], bins=12)
plt.title("Minutes Remaining Distribution")
plt.show()

✅ What it shows:
This tells us how many shots were taken with X minutes left in the quarter.
💡 Insights:
Spikes around 6–12 minutes are common.
Very few shots happen in the last minute, but those are often high-pressure shots.

Why it matters:
Timing plays a key role in the game. We’ll later combine this with seconds_remaining to create a time_remaining feature.

In [ ]:
# ✅ Seconds Remaining
sns.histplot(data['seconds_remaining'], bins=30)
plt.title("Seconds Remaining Distribution")
plt.show()

✅ What it shows:
This shows the distribution of shots taken during different seconds of a minute.
💡 Insights:
Sharp increase in shots during the last 5 seconds.
These are likely buzzer beaters or end-of-possession shots.

Why it matters:
We can create a clutch_moment feature: when time_remaining <= 5, scoring becomes less probable.

In [ ]:
# ✅ Playoff Distribution
sns.countplot(x='playoffs', data=data)
plt.title("Playoffs (1) vs Regular Season (0)")
plt.show()

✅ What it shows:
This tells how many shots were taken in playoffs (1) vs regular season (0).
💡 Insights:
Majority of shots are from the regular season.
Playoff shots are fewer but often happen in more intense, clutch conditions.

Why it matters:
Players tend to shoot differently under playoff pressure — this feature might influence the shot outcome.

##  Bivariate Analysis

In [ ]:
# ✅ Shot Accuracy vs Distance
sns.boxplot(x='shot_made_flag', y='shot_distance', data=data)
plt.title("Shot Accuracy vs Shot Distance")
plt.xlabel("Shot Made Flag")
plt.ylabel("Shot Distance")
plt.show()

Outlier Analysis: Shot Distance
- We observed several outliers in `shot_distance`, especially for missed shots.
- Most successful shots occur within 30 feet.
- Outliers mostly represent deep 3-pointers or half-court heaves.
- Depending on the model, we may choose to either retain or remove these for optimal performance.

In [ ]:
# Outlier Handling
Q1 = data['shot_distance'].quantile(0.25)
Q3 = data['shot_distance'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out extreme outliers
data = data[(data['shot_distance'] >= lower_bound) & (data['shot_distance'] <= upper_bound)]

✅ What it shows:
This boxplot compares the distribution of shot distances for made vs missed shots.
💡 Insights:
Made shots tend to have a lower median distance — makes sense, closer shots are easier.
Missed shots have a wider spread and more long-distance attempts.
Some successful long shots are outliers (possibly buzzer beaters or 3PTs).

Why it matters:
Shot distance is strongly predictive of whether a shot will be made.
This plot justifies keeping shot_distance as an important numerical feature.

In [ ]:
# ✅ Shot Zone vs Outcome
plt.figure(figsize=(12, 6))
sns.countplot(x='shot_zone_area', hue='shot_made_flag', data=data)
plt.title("Shot Zone Area vs Shot Outcome")
plt.xticks(rotation=45)
plt.show()

✅ What it shows:
A stacked countplot showing how many shots were made or missed in each zone area:
Left Side
Right Side
Center
Back Court, etc.
💡 Insights:
Shots from the center area generally have a better success rate.
Backcourt shots (long distance) are rarely made — low percentage.
Right vs Left side may show subtle player preferences.

Why it matters:
shot_zone_area is a categorical feature with location-based insights.
Helps in recommending better zones to shoot from.
May require encoding (e.g., one-hot or label encoding).

In [ ]:
# ✅ Shot Zone Basic
plt.figure(figsize=(12, 6))
sns.countplot(x='shot_zone_basic', hue='shot_made_flag', data=data)
plt.title("Basic Shot Zones vs Shot Outcome")
plt.xticks(rotation=45)
plt.show()

✅ What it shows:
A detailed breakdown of shots in specific basic court zones (like “Restricted Area”, “Mid-Range”, “In The Paint”, “Above the Break 3”, etc.) and whether they were made or missed.
💡 Insights:
Restricted Area (close to the basket) has a very high make rate.
Mid-range has more misses — least efficient in modern basketball.
Corner 3s often show a decent success rate (shortest 3PTs).

Why it matters:
shot_zone_basic is extremely important for feature selection.
Coaches use this insight to discourage mid-range shots and promote efficient zones.

## Heatmap of Shot Locations

In [ ]:
# ✅ KDE Plot of Shot Locations (Court Heatmap)
plt.figure(figsize=(10, 6))
sns.kdeplot(
    x=data['loc_x'],
    y=data['loc_y'],
    fill=True,
    cmap='viridis',
    thresh=0.05
)
plt.title("Heatmap of Shot Locations")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.show()

✅ What This Shows:
High-density areas (brighter/yellow) indicate most frequent shot locations.
You’ll likely see a dense cluster near the basket (layups, dunks).
A second cluster may appear around the 3-point line (especially above the arc).

💡 Insights You Can Mention:
Kobe took a large number of shots in the paint/restricted area.
Significant number of shots from top of the key (3-point arc).
Very few attempts from far corners and backcourt (those are desperation shots).

In [ ]:
# Convert float to int first (0.0/1.0 to 0/1)
data['shot_made_flag'] = data['shot_made_flag'].astype(int)

# Map 0 → 'Missed', 1 → 'Made'
data['shot_result'] = data['shot_made_flag'].map({0: 'Missed', 1: 'Made'})


🧠 Insight:
The original shot_made_flag column contains binary values in float format (0.0 for a missed shot and 1.0 for a made shot). These values are essential for training the model, but not very intuitive when analyzing the data visually or sharing with stakeholders.
✅ Therefore:
First, we convert the column from float to int to ensure clean binary classification (0 or 1).
Then, we map the numeric values to their corresponding labels using a dictionary:
0 → 'Missed'
1 → 'Made'

In [ ]:
#unique data in output column
print(data['shot_result'].unique()) 

In [ ]:
# Made vs Missed Shots
import plotly.express as px

fig = px.scatter(
    data,
    x='loc_x',
    y='loc_y',
    color='shot_result',  # Now categorical
    title='🎯 Kobe Bryant Shot Chart (Made vs Missed)',
    labels={'loc_x': 'Court X', 'loc_y': 'Court Y', 'shot_result': 'Shot Outcome'},
    opacity=0.5,
    color_discrete_map={'Made': 'green', 'Missed': 'red'}  # Optional: nice color scheme
)

fig.update_layout(height=600, width=1000)
fig.show()



✅ What it shows:
This interactive scatter plot is based on court location and shot outcome.
Each dot represents one shot.
Green = Made, Red = Missed.
X/Y axes represent the court layout.
The interactive plot allows you to zoom in, hover, and explore specific regions (e.g., paint, 3-point line, corners).

Key Insights:
🔹 1. High Accuracy in Restricted Area
There is a dense green cluster near the basket (loc_y ≈ 0 to 100).
Kobe had a very high success rate close to the rim — layups, dunks, and short jumpers.
🔹 2. More Misses From Long Distance
As the distance from the basket increases (loc_y > 200), red dots increase.
Long 2s and deep 3s had lower accuracy — expected due to shot difficulty.
🔹 3. Balanced Shooting Behavior
The chart is fairly symmetrical along the X-axis, meaning Kobe shot from both left and right wings regularly.
🔹 4. Fewer Corner Shots
Few points near loc_x = ±220, loc_y < 100 — corners of the court.
Indicates that Kobe preferred center and wing areas over corners.

Short insight : Kobe’s most efficient zones were close to the basket. Coaches could use this insight to optimize offensive plays that bring him closer to the rim while limiting long 2-point attempts.

## Feature Engineering 

Feature engineering is the process of selecting, transforming, and creating new features from raw data to improve the performance of machine learning models

In [ ]:
# feature vs why it's Useful to merge the columns

In [ ]:
data['time_remaining'] = data['minutes_remaining'] * 60 + data['seconds_remaining']

Insight : time_remaining : Combines minutes_remaining + seconds_remaining to show exact time left,Total seconds left in the quarter when the shot was taken.

In [ ]:
data['is_clutch'] = data['time_remaining'].apply(lambda x: 1 if x <= 5 else 0)

Insight : is_clutch : Flags shots in the last 5 seconds of a quarter (high-pressure situations),Whether the shot was taken under pressure (≤ 5 seconds left in the quarter).
1 = clutch shot, 0 = not clutch

In [ ]:
data['is_3pt'] = data['shot_type'].apply(lambda x: 1 if '3PT' in x else 0)

Insight : Whether the shot was a 3-point attempt.
1 = 3PT, 0 = 2PT

In [ ]:
# home_game Indicates home vs away games (matchup field)
data['home_game'] = data['matchup'].apply(lambda x: 1 if 'vs' in x else 0)

Insight : Whether the game was played at home.
1 = home (matchup contains 'vs'), 0 = away (matchup contains '@')

In [ ]:
# season_start Extracts year from season (e.g., '2002-03' → 2002)
data['season_start'] = data['season'].apply(lambda x: int(x.split('-')[0]))

Insight : The starting year of the NBA season the shot was taken in.
e.g., '2000-01' becomes 2000

In [ ]:
data[['time_remaining', 'is_clutch', 'is_3pt', 'home_game', 'season_start']].head()

In [ ]:
# Accuracy of Shot in Clutch vs Non-Clutch Moment
sns.barplot(x='is_clutch', y='shot_made_flag', data=data)
plt.title('Shot Accuracy in Clutch vs Non-Clutch')
plt.xticks([0,1], ['Non-Clutch', 'Clutch'])
plt.ylabel('Shot Success Rate')
plt.show()

## Data Preprocessing

It involves transforming raw data into a format that is suitable for analysis and modeling by cleaning, normalizing, and transforming it

✅ This step includes:
Dropping unnecessary columns
Encoding categorical variables
Feature scaling
Splitting into train-test sets

In [ ]:
# Step 1 : Drop Unnecessary Columns

In [ ]:
# Drop columns that are IDs or not predictive
columns_to_drop = [
    'shot_id', 'game_id', 'game_event_id',
    'team_id', 'team_name', 'game_date',
    'matchup', 'opponent', 'season',
    'minutes_remaining', 'seconds_remaining',
    'shot_result'
]
data = data.drop(columns=columns_to_drop)

In [ ]:
# Step 2 :  encode Categorical Variables

What Is One-Hot Encoding?
One-hot encoding converts a categorical feature (text-based or label-based) into binary columns (0/1) — one for each unique category.

Why We Need One-Hot Encoding
Logistic Regression assumes linear relationships between features and the target.
If we used LabelEncoder, we'd be assigning integers to categories, which would falsely imply order/rank (e.g., 'Jump Shot' > 'Dunk').
One-hot encoding removes this bias by treating each category independently.

✅ columns needed for one hot encoding and why 
action_type : High-cardinality text (50+ types) , eg : 'Jump Shot', 'Layup Shot', 'Slam Dunk'
combined_shot_type	Few categories but text-based	eg : '2PT Field Goal', '3PT Field Goal'
shot_type	Redundant with combined, but still text	eg : '2PT Field Goal', '3PT Field Goal'
shot_zone_area	Categorical spatial zone	eg : 'Center(C)', 'Right Side(C)'
shot_zone_basic	Court area breakdown	eg : 'Restricted Area', 'Mid-Range'
shot_zone_range	Distance buckets	eg : 'Less Than 8 ft.', '24+ ft.'

In [ ]:
# List of categorical columns to one-hot encode
cat_cols = ['action_type', 'combined_shot_type', 'shot_type',
            'shot_zone_area', 'shot_zone_basic', 'shot_zone_range']

# Apply one-hot encoding
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)


This transformation was necessary to convert categorical values into numeric format without introducing artificial order, especially for Logistic Regression. Each category is now represented as a separate binary column (0/1).

## Model Training

🎯 Goal:
Train and evaluate multiple ML models to predict if Kobe’s shot will be made or missed (shot_made_flag) based on shot context.

In [ ]:
# Split into Features (X) and Target (y)

In [ ]:
# Target variable
y = data['shot_made_flag'].astype(int)
# Features
X = data.drop(columns=['shot_made_flag'])

In [ ]:
# Scale Numerical Features : 'lat', 'lon', 'loc_x', 'loc_y', 'shot_distance', 'time_remaining'

In [ ]:
to_scale = ['lat', 'lon', 'loc_x', 'loc_y', 'shot_distance', 'time_remaining']

scaler = StandardScaler()

X[to_scale] = scaler.fit_transform(X[to_scale])

In [ ]:
# Train-Test Split : Split your data into 80% training, 20% testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

stratify=y ensures that:
The distribution of the target variable (y) is maintained in both:
Training set
Testing set

Why Is This Important?
Let’s say your original target (shot_made_flag) looks like this:
60% missed (0)
40% made (1)
Without stratify=y:
Your train/test split might be imbalanced by chance (e.g., test set might have only 30% made shots).
With stratify=y:
Both train and test sets will have about 60/40 split — just like the original dataset.

## Logistic Regression

In [ ]:
# 1. Define parameter grid
log_params = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear']
}

In [ ]:
# 2. Stratified Cross Validation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# 3. Grid Search
log_grid = GridSearchCV(LogisticRegression(max_iter=1000), log_params, cv=cv_strategy, scoring='f1', n_jobs=-1)
log_grid.fit(X_train, y_train)

In [ ]:
# 4. Best model
best_log = log_grid.best_estimator_

In [ ]:
# 5. Predictions
y_pred_log = best_log.predict(X_test)
y_prob_log = best_log.predict_proba(X_test)[:, 1]

In [ ]:
# 6. Evaluation
print("Best Params:", log_grid.best_params_)
print("Accuracy :", accuracy_score(y_test, y_pred_log))
print("F1 Score :", f1_score(y_test, y_pred_log))
print("ROC AUC  :", roc_auc_score(y_test, y_prob_log))

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_log)

# Plot
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Missed', 'Pred Made'], yticklabels=['Actual Missed', 'Actual Made'])
plt.title("Logistic Regression - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Confusion Matrix values
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_log).ravel()

# Manual calculations
accuracy  = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1_score_ = 2 * (precision * recall) / (precision + recall)

print(f"Manual Accuracy : {accuracy:.3f}")
print(f"Manual Precision: {precision:.3f}")
print(f"Manual Recall   : {recall:.3f}")
print(f"Manual F1 Score : {f1_score_:.3f}")


# Insight:
Precision answers: “Of all predicted made shots, how many were actually made?”
Recall answers: “Of all actually made shots, how many did we correctly predict?”
F1 is the balanced performance measure between both.

Algorithm Report :  Logistic Regression
Best Params: C=0.01, penalty='l2', solver='liblinear'
Accuracy: 68.05%, F1 Score: 0.571, ROC AUC: 0.697
🔍 Insight:
Logistic Regression performed decently, indicating linear boundaries exist in the data.
High regularization (C=0.01) helped generalize, but may have limited its flexibility.
It maintained decent ranking ability (ROC AUC ~ 0.70) despite moderate F1.